# The TabPFN Dimensionality Wall (Group 3.3): 
> Track average accuracy across datasetsfiltered by d < 50, d = 100, and d > 100. This isolates your main tuning-free competitor,demonstrating its hard architectural limits while SmallMLP scales gracefully.



In [ ]:
import os
from pathlib import Path
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# 0. Load TABPFN_TOKEN (and optional HF_TOKEN) from root .env before importing tabpfn
root_env = Path.cwd().parent / ".env"
assert root_env.exists(), f"Root .env not found at {root_env}"
load_dotenv(dotenv_path=root_env, override=True)

tabpfn_token = os.getenv("TABPFN_TOKEN", "").strip()
assert tabpfn_token.startswith(
    "tabpfn_sk_"
), f"Invalid or empty TABPFN_TOKEN in {root_env}"
os.environ["TABPFN_TOKEN"] = tabpfn_token

if os.getenv("HF_TOKEN"):
  os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN").strip()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tabpfn import TabPFNClassifier
import xgboost as xgb

from smallgbm import SmallGBMClassifier
from smallmlp import SmallMLPClassifier


# 1. Helper to load strictly disjoint Hold-Out OpenML/UCI datasets
def load_openml_clean(data_id):
  ds = fetch_openml(data_id=data_id, as_frame=True, parser="auto")
  X = pd.get_dummies(ds.data, drop_first=True).values.astype(float)
  X = SimpleImputer(strategy="median").fit_transform(X)
  y = LabelEncoder().fit_transform(ds.target.astype(str))
  return X, y


# Hold-Out Classification Datasets from Section 3.1 (No sklearn toy datasets)
holdout_curves_sets = {
    "Hepatitis": load_openml_clean(55),  # n=155, d=19
    "Parkinsons": load_openml_clean(1488),  # n=195, d=22
    "Ionosphere": load_openml_clean(59),  # n=351, d=34
    "BreastCancer_UCI": load_openml_clean(1510),  # n=569, d=30 (WDBC)
}

target_dims = [35, 50, 80, 100, 120, 150, 200, 300]
models = [
    "SmallMLP",
    "SmallGBM",
    "TabPFN (PCA-100 >100d)",
    "TabPFN (Truncate-100 >100d)",
    "XGBoost (Default)",
    "Random Forest",
]

# Pre-warm TabPFN checkpoint once so weights are cached locally
_warmup = TabPFNClassifier(device="cpu", n_estimators=4)
_warmup.fit(np.random.randn(20, 4), np.array([0, 1] * 10))
print("TabPFN checkpoint cached and ready.")

# Store per-dataset mean accuracies to compute mean & std across the 4 datasets
per_ds_results = {
    m: np.zeros((len(holdout_curves_sets), len(target_dims))) for m in models
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for ds_idx, (ds_name, (X_base, y)) in enumerate(holdout_curves_sets.items()):
  n_samples, d_base = X_base.shape
  print(f"\n--- Running Hold-Out Dataset: {ds_name} (n={n_samples}, d={d_base}) ---")

  for idx_d, d_target in enumerate(target_dims):
    rng = np.random.RandomState(42 + d_target)
    X_std_base = StandardScaler().fit_transform(X_base)

    if d_target > d_base:
      # Inject correlated projections + Gaussian noise and shuffle columns
      W = rng.normal(0, 0.5, size=(d_base, d_target - d_base))
      corr_noise = X_std_base @ W + rng.normal(
          0, 1.0, size=(n_samples, d_target - d_base)
      )
      X = np.hstack([X_std_base, corr_noise])
      X = X[:, rng.permutation(d_target)]
    else:
      X = X_std_base.copy()

    fold_scores = {m: [] for m in models}

    for train_idx, test_idx in skf.split(X, y):
      X_tr, X_te = X[train_idx], X[test_idx]
      y_tr, y_te = y[train_idx], y[test_idx]

      scaler = StandardScaler()
      X_tr_s = scaler.fit_transform(X_tr)
      X_te_s = scaler.transform(X_te)

      # 1. SmallMLP (frozen alpha=6.0, beta=0.7)
      mlp = SmallMLPClassifier(alpha=6.0, beta=0.7, random_state=42)
      mlp.fit(X_tr_s, y_tr)
      fold_scores["SmallMLP"].append(accuracy_score(y_te, mlp.predict(X_te_s)))

      # 2. SmallGBM
      sgbm = SmallGBMClassifier(random_state=42)
      sgbm.fit(X_tr, y_tr)
      fold_scores["SmallGBM"].append(accuracy_score(y_te, sgbm.predict(X_te)))

      # 3. TabPFN with PCA-100 projection when d > 100
      pfn_pca = TabPFNClassifier(device="cpu", n_estimators=4)
      if d_target > 100:
        pca = PCA(n_components=100, random_state=42)
        X_tr_p = pca.fit_transform(X_tr_s)
        X_te_p = pca.transform(X_te_s)
      else:
        X_tr_p, X_te_p = X_tr_s, X_te_s
      pfn_pca.fit(X_tr_p, y_tr)
      fold_scores["TabPFN (PCA-100 >100d)"].append(
          accuracy_score(y_te, pfn_pca.predict(X_te_p))
      )

      # 4. TabPFN with 100-feature truncation/subsampling when d > 100
      pfn_sub = TabPFNClassifier(device="cpu", n_estimators=4)
      X_tr_sub = X_tr_s[:, :100] if d_target > 100 else X_tr_s
      X_te_sub = X_te_s[:, :100] if d_target > 100 else X_te_s
      pfn_sub.fit(X_tr_sub, y_tr)
      fold_scores["TabPFN (Truncate-100 >100d)"].append(
          accuracy_score(y_te, pfn_sub.predict(X_te_sub))
      )

      # 5. XGBoost Default
      model_xgb = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
      model_xgb.fit(X_tr, y_tr)
      fold_scores["XGBoost (Default)"].append(
          accuracy_score(y_te, model_xgb.predict(X_te))
      )

      # 6. Random Forest
      rf = RandomForestClassifier(n_estimators=100, random_state=42)
      rf.fit(X_tr, y_tr)
      fold_scores["Random Forest"].append(
          accuracy_score(y_te, rf.predict(X_te))
      )

    for m in models:
      per_ds_results[m][ds_idx, idx_d] = np.mean(fold_scores[m])

    print(
        f"d={d_target:3d} | SmallMLP: {per_ds_results['SmallMLP'][ds_idx, idx_d]:.4f} | "
        f"SmallGBM: {per_ds_results['SmallGBM'][ds_idx, idx_d]:.4f} | "
        f"TabPFN-PCA: {per_ds_results['TabPFN (PCA-100 >100d)'][ds_idx, idx_d]:.4f} | "
        f"TabPFN-Trunc: {per_ds_results['TabPFN (Truncate-100 >100d)'][ds_idx, idx_d]:.4f}"
    )

# 2. Aggregate across all 4 hold-out datasets and save CSV
mean_results = {m: np.mean(per_ds_results[m], axis=0) for m in models}
sem_results = {
    m: np.std(per_ds_results[m], axis=0) / np.sqrt(len(holdout_curves_sets))
    for m in models
}

df_curve = pd.DataFrame(mean_results, index=target_dims)
df_curve.index.name = "Dimensionality (d)"
print("\n=== Mean Accuracy Across 4 Hold-Out Datasets ===")
print(df_curve.round(4))
df_curve.to_csv("tabpfn_dim_wall_4datasets.csv")

# 3. Publication-grade plot (ICLR/NeurIPS formatting: no top title, clear legend)
plt.figure(figsize=(7.5, 4.8), dpi=300)

styles = {
    "SmallMLP": ("#1f77b4", "o", "-", 2.5),
    "SmallGBM": ("#2ca02c", "s", "-", 2.2),
    "TabPFN (PCA-100 >100d)": ("#d62728", "^", "--", 2.2),
    "TabPFN (Truncate-100 >100d)": ("#9467bd", "P", "--", 2.0),
    "XGBoost (Default)": ("#7f7f7f", "d", ":", 1.8),
    "Random Forest": ("#ff7f0e", "v", "-.", 1.8),
}

for name, (color, marker, ls, lw) in styles.items():
  means = mean_results[name]
  plt.plot(
      target_dims,
      means,
      label=name,
      color=color,
      marker=marker,
      linestyle=ls,
      linewidth=lw,
      markersize=6,
  )

# Highlight the d=100 architectural boundary
plt.axvline(x=100, color="#d62728", linestyle=":", linewidth=1.8, alpha=0.85)
ymin, ymax = plt.ylim()
plt.text(
    104,
    ymin + 0.015 * (ymax - ymin),
    "TabPFN Native Ceiling ($d=100$)\n(Projection / Subsampling Regime $\\rightarrow$)",
    color="#d62728",
    fontsize=10,
    fontweight="bold",
)

plt.xlabel(r"Feature Dimensionality ($d$)", fontsize=12, labelpad=8)
plt.ylabel(
    "Mean 5-Fold Test Accuracy (4 Hold-Out Sets)", fontsize=12, labelpad=8
)
plt.xticks(target_dims, fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.35)
plt.legend(frameon=True, fontsize=9, loc="lower left")

plt.tight_layout()
plt.savefig("tabpfn_dim_wall.pdf", bbox_inches="tight")
plt.savefig("tabpfn_dim_wall.png", dpi=300, bbox_inches="tight")
plt.show()